# A Unifying Theory of High-Latitude Geophysical Phenomena (Axford & Hines, 1961): Implementation / 구현

Axford-Hines의 점성 상호작용 모델을 구현하고, Dungey 모델과의 비교를 통해 두 패러다임의 차이와 공통점을 시각화합니다.

Implementing the Axford-Hines viscous interaction model and comparing with Dungey's model to visualize the differences and commonalities of both paradigms.

**구현 내용 / Implementation:**
1. 쌍세포 대류 패턴: Dungey vs. Axford-Hines 비교
2. 공전(Corotation) vs. 대류(Convection): 플라즈마권 경계
3. 점성 vs. 재결합: 에너지 기여도 비교
4. 종합 — 현대 자기권 대류 모델

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import Normalize

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

R_E = 6.371e6  # Earth radius (m)
Omega_E = 7.292e-5  # Earth rotation rate (rad/s)
B0 = 3.1e-5  # Equatorial surface field (T)

print("Constants loaded / 상수 로드 완료")

## Part 1: 쌍세포 대류 패턴 — Dungey vs. Axford-Hines / Two-Cell Convection — Dungey vs. Axford-Hines

두 모델 모두 극관에서 반태양 방향, 저위도에서 태양 방향의 **쌍세포 대류**를 예측합니다. 동력원은 다르지만 결과 패턴은 거의 동일합니다.

Both models predict antisunward polar cap flow and sunward return at lower latitudes — **two-cell convection**. Different drivers, nearly identical patterns.

- **Dungey**: 열린 자기장선의 이동 → 극관 전위 = $v_{SW} \cdot |B_z| \cdot L_{\text{eff}}$
- **Axford-Hines**: 경계층 점성력 → 극관 전위 = $v_{\text{conv}} \cdot B \cdot L$

In [ ]:
def convection_potential(colat_deg, mlt_hr, phi_total=60.0, colat_boundary=15.0):
    """Two-cell convection potential in the ionosphere.

    Args:
        colat_deg: Magnetic colatitude from pole (degrees).
        mlt_hr: Magnetic local time (hours).
        phi_total: Total cross-polar-cap potential (kV).
        colat_boundary: Open/closed boundary colatitude (degrees).

    Returns:
        Potential (kV).
    """
    mlt_rad = mlt_hr * np.pi / 12.0
    dawn_dusk = np.sin(mlt_rad)

    potential = np.zeros_like(colat_deg, dtype=float)
    inside = colat_deg < colat_boundary

    # Inside polar cap: linear potential
    potential[inside] = (phi_total / 2) * dawn_dusk[inside] * (colat_deg[inside] / colat_boundary)

    # Outside: reversed, decaying
    outside = ~inside
    decay = np.exp(-(colat_deg[outside] - colat_boundary) / 10.0)
    potential[outside] = -(phi_total / 2) * dawn_dusk[outside] * decay * 0.5

    return potential


# Create polar plots comparing the two models
fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(polar=True))

colat = np.linspace(0, 50, 200)
mlt = np.linspace(0, 24, 200)
COLAT, MLT = np.meshgrid(colat, mlt)
theta_plot = MLT * np.pi / 12.0
r_plot = COLAT

scenarios = [
    ("Dungey Only\n(Reconnection, Bz=-10 nT)\nDungey 재결합만", 50.0),
    ("Axford-Hines Only\n(Viscous interaction)\n점성 상호작용만", 15.0),
    ("Combined (Modern)\n(Reconnection + Viscous)\n현대 결합 모델", 65.0),
]

for ax, (title, phi_pc) in zip(axes, scenarios):
    phi = convection_potential(COLAT, MLT, phi_total=phi_pc, colat_boundary=15.0)

    levels = np.linspace(-35, 35, 25)
    cs = ax.contour(theta_plot, r_plot, phi, levels=levels, cmap="RdBu_r", linewidths=1.0)

    ax.set_theta_zero_location("S")
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 40)
    ax.set_yticks([10, 20, 30, 40])
    ax.set_yticklabels(["80°", "70°", "60°", "50°"], fontsize=7)

    # Auroral oval
    theta_oval = np.linspace(0, 2 * np.pi, 100)
    ax.plot(theta_oval, np.full_like(theta_oval, 15), "k-", lw=2)

    for hr, label in [(0, "00"), (6, "06"), (12, "12"), (18, "18")]:
        ax.text(hr * np.pi / 12, 45, label, ha="center", fontsize=8)

    ax.set_title(f"{title}\n$\\Phi_{{PC}}$ = {phi_pc:.0f} kV", fontsize=10, pad=15)

plt.suptitle("Two-Cell Convection: Dungey vs. Axford-Hines vs. Combined\n"
             "쌍세포 대류: Dungey vs. Axford-Hines vs. 결합 모델", fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

print("Key insight: SAME convection pattern, DIFFERENT drivers!")
print("핵심: 같은 대류 패턴, 다른 동력원!")
print(f"  Dungey contribution: ~50 kV (reconnection, Bz=-10 nT)")
print(f"  Axford-Hines contribution: ~15 kV (viscous)")
print(f"  Total: ~65 kV (combined)")

## Part 2: 공전 vs. 대류 — 플라즈마권 경계 / Corotation vs. Convection — Plasmapause

Axford-Hines가 처음 명확히 논의한 핵심 물리: 지구 자전에 의한 **공전(corotation)**과 태양풍에 의한 **대류(convection)**의 경쟁. 두 힘이 균형을 이루는 곳이 **플라즈마권 경계(plasmapause)**를 정의합니다.

Core physics first clearly discussed by Axford-Hines: competition between **corotation** (from Earth's rotation) and **convection** (from solar wind). The balance defines the **plasmapause**.

- 공전 전위: $\Phi_{\text{corot}} = -\Omega B_0 R_E^2 / L$
- 대류 전위: $\Phi_{\text{conv}} = -\Phi_{PC} \sin(\phi) / 2$ (새벽-황혼 비대칭)
- 균형점: 플라즈마권 경계 (Alfvén layer)

In [ ]:
def corotation_potential(L: np.ndarray) -> np.ndarray:
    """Corotation electric potential in the equatorial plane.

    Phi_corot = -Omega * B0 * R_E^2 / L (in SI, gives Volts)

    Args:
        L: L-shell parameter (equatorial distance in R_E).

    Returns:
        Potential (kV).
    """
    return -Omega_E * B0 * R_E**2 / (L * R_E) / 1e3  # kV


def convection_potential_equatorial(
    L: np.ndarray,
    phi_rad: np.ndarray,
    phi_pc_kV: float = 60.0,
) -> np.ndarray:
    """Uniform dawn-dusk convection electric potential in equatorial plane.

    Volland-Stern model: Phi_conv = -A * L^gamma * sin(phi)
    Simplified: Phi_conv = -(Phi_PC/2) * (L/L_max)^2 * sin(phi)

    Args:
        L: L-shell (R_E).
        phi_rad: Azimuthal angle (0=noon, pi/2=dusk, pi=midnight).
        phi_pc_kV: Cross-polar-cap potential (kV).

    Returns:
        Potential (kV).
    """
    L_max = 10.0  # Approximate magnetopause distance
    return -(phi_pc_kV / 2) * (L / L_max) ** 2 * np.sin(phi_rad)


def total_potential(L, phi_rad, phi_pc_kV=60.0):
    """Total potential = corotation + convection."""
    return corotation_potential(L) + convection_potential_equatorial(L, phi_rad, phi_pc_kV)


# --- Equatorial plane potential map ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

L_range = np.linspace(1.01, 10, 300)
phi_range = np.linspace(0, 2 * np.pi, 300)
L_grid, PHI_grid = np.meshgrid(L_range, phi_range)

X_eq = L_grid * np.cos(PHI_grid)
Y_eq = L_grid * np.sin(PHI_grid)

panels = [
    ("Corotation Only\n공전만", lambda L, p: corotation_potential(L)),
    ("Convection Only ($\\Phi_{PC}$=60 kV)\n대류만", lambda L, p: convection_potential_equatorial(L, p, 60)),
    ("Total (Corotation + Convection)\n총 전위 (공전 + 대류)", lambda L, p: total_potential(L, p, 60)),
]

for ax, (title, pot_func) in zip(axes, panels):
    pot = pot_func(L_grid, PHI_grid)

    levels = np.linspace(-100, 40, 30)
    cs = ax.contour(X_eq, Y_eq, pot, levels=levels, cmap="RdBu_r", linewidths=0.8)

    earth = Circle((0, 0), 1.0, facecolor="steelblue", edgecolor="black", lw=2, zorder=10)
    ax.add_patch(earth)
    ax.text(0, 0, "E", ha="center", va="center", fontsize=9,
            fontweight="bold", color="white", zorder=11)

    # Find plasmapause on dusk side (phi = pi/2)
    if "Total" in title:
        L_test = np.linspace(2, 8, 1000)
        pot_dusk = total_potential(L_test, np.full_like(L_test, np.pi / 2), 60)
        # Plasmapause: where total potential has a stagnation point
        # (maximum of total potential on dusk side)
        idx_max = np.argmax(pot_dusk)
        L_pp = L_test[idx_max]

        # Draw plasmapause as last closed equipotential
        pp_theta = np.linspace(0, 2 * np.pi, 200)
        pot_pp = pot_dusk[idx_max]
        # Find L at this potential for each angle
        L_pp_curve = []
        for th in pp_theta:
            pot_at_th = total_potential(L_test, np.full_like(L_test, th), 60)
            # Find where potential crosses pot_pp
            crossings = np.where(np.diff(np.sign(pot_at_th - pot_pp)))[0]
            if len(crossings) > 0:
                L_pp_curve.append(L_test[crossings[0]])
            else:
                L_pp_curve.append(np.nan)
        L_pp_arr = np.array(L_pp_curve)
        valid = ~np.isnan(L_pp_arr)
        ax.plot(L_pp_arr[valid] * np.cos(pp_theta[valid]),
                L_pp_arr[valid] * np.sin(pp_theta[valid]),
                "lime", lw=3, ls="--", label=f"Plasmapause (~{L_pp:.1f} $R_E$ dusk)")
        ax.legend(fontsize=8, loc="lower left")

    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect("equal")
    ax.set_xlabel("X (sunward →) [$R_E$]")
    ax.set_ylabel("Y (duskward ↑) [$R_E$]")
    ax.set_title(title, fontsize=10)

    # Direction labels
    ax.text(9, 0, "Sun\n태양", fontsize=7, ha="center", color="orange")
    ax.text(0, 9, "Dusk\n황혼", fontsize=7, ha="center", color="purple")

plt.suptitle("Equatorial Plane Potentials: Corotation vs. Convection\n"
             "적도면 전위: 공전 vs. 대류", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Part 3: 점성 vs. 재결합 에너지 기여도 / Viscous vs. Reconnection Energy Contribution

현대적 이해: Dungey 재결합이 ~70–90%, Axford-Hines 점성이 ~10–30%의 에너지를 전달합니다. 그러나 IMF 방향에 따라 비율이 크게 변합니다.

Modern understanding: Dungey reconnection delivers ~70–90%, Axford-Hines viscosity ~10–30% of energy. But the ratio varies strongly with IMF direction.

In [ ]:
def phi_dungey(bz_nT, v_sw=400, L_eff_RE=7.0):
    """Dungey reconnection contribution to Phi_PC (kV)."""
    if bz_nT >= 0:
        return 0.0  # No reconnection for northward IMF
    return abs(bz_nT) * 1e-9 * v_sw * 1e3 * L_eff_RE * R_E / 1e3

def phi_viscous(v_sw=400, efficiency=0.05):
    """Axford-Hines viscous contribution to Phi_PC (kV).

    Approximately proportional to v_sw, independent of IMF.
    """
    return efficiency * v_sw * 5e-9 * 7.0 * R_E / 1e3  # ~15 kV at 400 km/s


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Phi_PC contributions vs. IMF Bz
ax = axes[0]
bz_scan = np.linspace(-20, 10, 200)
phi_d = np.array([phi_dungey(bz) for bz in bz_scan])
phi_v = np.array([phi_viscous() for _ in bz_scan])
phi_tot = phi_d + phi_v

ax.fill_between(bz_scan, 0, phi_v, color="orange", alpha=0.3, label="Viscous (A-H)\n점성 (A-H)")
ax.fill_between(bz_scan, phi_v, phi_tot, color="blue", alpha=0.3, label="Reconnection (Dungey)\n재결합 (Dungey)")
ax.plot(bz_scan, phi_tot, "k-", lw=2, label="Total / 합계")
ax.plot(bz_scan, phi_v, "orange", lw=1.5, ls="--")

ax.axvline(0, color="gray", ls=":", alpha=0.5)
ax.text(-15, 75, "Southward\n남향", fontsize=9, color="red")
ax.text(3, 75, "Northward\n북향", fontsize=9, color="blue")

# Mark key: when Bz > 0, only viscous works
ax.annotate("Only viscous!\n점성만!", xy=(5, phi_viscous()), xytext=(7, 35),
            fontsize=9, color="orange", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="orange"))

ax.set_xlabel("IMF $B_z$ (nT)")
ax.set_ylabel("$\\Phi_{PC}$ (kV)")
ax.set_title("(a) Energy Contributions vs. IMF $B_z$\n에너지 기여도 vs. IMF $B_z$")
ax.legend(fontsize=8)
ax.set_ylim(0, 100)

# (b) Pie chart for typical conditions
ax = axes[1]
conditions = [
    ("Southward\n남향\n(Bz=-10 nT)", phi_dungey(-10), phi_viscous()),
    ("Northward\n북향\n(Bz=+5 nT)", phi_dungey(5), phi_viscous()),
]

for idx, (label, d, v) in enumerate(conditions):
    total = d + v
    sizes = [d / total * 100 if total > 0 else 0, v / total * 100 if total > 0 else 100]
    colors_pie = ["steelblue", "orange"]
    wedges, texts, autotexts = ax.pie(
        sizes if idx == 0 else [],  # Only show first pie
        labels=["Reconnection", "Viscous"] if idx == 0 else [],
        colors=colors_pie,
        autopct="%1.0f%%" if idx == 0 else "",
        startangle=90,
        radius=1 - idx * 0.4,
    )

# Manual for southward
ax.clear()
d1, v1 = phi_dungey(-10), phi_viscous()
t1 = d1 + v1
d2, v2 = phi_dungey(5), phi_viscous()
t2 = d2 + v2

x = np.arange(2)
width = 0.35
bars_d = ax.bar(x, [d1, d2], width, label="Reconnection\n재결합", color="steelblue")
bars_v = ax.bar(x, [v1, v2], width, bottom=[d1, d2], label="Viscous\n점성", color="orange")
ax.set_xticks(x)
ax.set_xticklabels(["Southward\n남향 (Bz=-10)", "Northward\n북향 (Bz=+5)"])
ax.set_ylabel("$\\Phi_{PC}$ (kV)")
ax.set_title("(b) Contribution Breakdown\n기여도 분해")
ax.legend(fontsize=9)

# Add percentage labels
for i, (d, v) in enumerate([(d1, v1), (d2, v2)]):
    total = d + v
    if d > 0:
        ax.text(i, d / 2, f"{d:.0f} kV\n({d/total*100:.0f}%)", ha="center", fontsize=8, color="white")
    ax.text(i, d + v / 2, f"{v:.0f} kV\n({v/total*100:.0f}%)", ha="center", fontsize=8)

# (c) Historical timeline of understanding
ax = axes[2]
ax.axis("off")
timeline = """
   Two Paradigms of Magnetospheric Physics
   자기권 물리학의 2대 패러다임

   1961  Dungey: Reconnection (open)
         Axford-Hines: Viscous (closed)
         → Same convection, different drivers

   1970s Observations suggest Dungey dominant
         but residual convection during Bz>0

   1975  Burton: Dst ∝ Bz (confirms Dungey)

   1980s KH instability at flanks identified
         → Specific viscous mechanism found

   2000s MMS/Cluster: Both mechanisms confirmed
         Reconnection ~70-90%, Viscous ~10-30%

   Modern: Both essential. Viscous provides
   "baseline" convection; reconnection provides
   "storm-time" enhancement.

   현대: 두 메커니즘 모두 필수.
   점성 = 기본 대류, 재결합 = 폭풍시 강화
"""
ax.text(0.05, 0.95, timeline, transform=ax.transAxes, fontsize=8.5,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.tight_layout()
plt.show()

## Summary / 요약

| Part | 구현 내용 / Implementation | Axford-Hines (1961)의 기여 / Contribution |
|------|--------------------------|------------------------------------------|
| 1 | 쌍세포 대류: Dungey vs. A-H vs. 결합 | 동일한 대류 패턴, 다른 동력원 / Same pattern, different drivers |
| 2 | 공전 vs. 대류: 플라즈마권 경계 | 두 힘의 경쟁이 plasmapause를 정의 / Competition defines plasmapause |
| 3 | 점성 vs. 재결합 에너지 기여도 | 현대: 재결합 ~70-90%, 점성 ~10-30% / Modern: reconnection ~70-90%, viscous ~10-30% |

**핵심 비교 / Key Comparison:**

| | Dungey (Paper #6) | Axford-Hines (이 논문 / This paper) |
|---|---|---|
| 자기권 상태 | 열린 (open) | 닫힌 (closed) |
| 동력원 | 자기 재결합 | 점성 운동량 전달 |
| IMF 의존 | $B_z < 0$ 필요 | IMF 무관 |
| 기여도 | ~70–90% | ~10–30% |
| 역할 | 폭풍시 주도 | 기본 대류 (baseline) |

**다음 논문 / Next paper**: #8 Akasofu (1964) — Substorm 현상학. Dungey cycle과 Axford-Hines 대류의 **시간 변동** 관점에서 오로라 substorm의 성장-팽창-회복 단계를 정의. / Defines substorm phases (growth-expansion-recovery) from the perspective of **temporal variation** of Dungey cycle and Axford-Hines convection.